# Cloud throughput gate (SPEC §11)

Measures total SAC steps/s and RAM on this machine so E2/E5's 1M-step runs can be split between the laptop and the cloud.

**Before running:**
- **Colab:** put `ur5rl_bundle.zip` (from `python scripts/make_bundle.py`) in `MyDrive/mtp/`. Runtime: CPU.
- **Kaggle:** add the zip as a dataset and attach it to this notebook. Accelerator: None. Internet: on (for pip).

Takes ~15–20 min (one run, one HER run, then `cores // 2` parallel runs). Send back the `SUMMARY` line.

In [ ]:
import glob, os, sys
PLATFORM = 'kaggle' if os.path.exists('/kaggle') else ('colab' if os.path.exists('/content') else 'other')
if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    BUNDLE, WORK = '/content/drive/MyDrive/mtp/ur5rl_bundle.zip', '/content/ur5rl'
elif PLATFORM == 'kaggle':
    BUNDLE, WORK = glob.glob('/kaggle/input/**/ur5rl_bundle.zip', recursive=True)[0], '/kaggle/working/ur5rl'
else:
    BUNDLE, WORK = 'ur5rl_bundle.zip', os.path.abspath('ur5rl')
print(PLATFORM, BUNDLE, WORK, sys.version)

In [ ]:
!rm -rf {WORK} && mkdir -p {WORK} && unzip -q {BUNDLE} -d {WORK}
%cd {WORK}
!pip install -q --index-url https://download.pytorch.org/whl/cpu torch==2.14.1
!pip install -q mujoco==3.17.0 gymnasium==1.4.0 stable-baselines3==2.9.2 numpy==2.4.6 scipy==1.17.1 PyYAML==6.0.3 psutil==7.2.2 tensorboard==2.21.0 pytest==9.1.1
!pip install -q -e . --no-deps --ignore-requires-python

In [ ]:
# sanity: the scene, FK, clearance and action rule behave the same on Linux
!python -m pytest -q tests/test_fk_tcp.py tests/test_clearance.py tests/test_action.py

In [ ]:
!python scripts/cloud_bench.py --platform {PLATFORM}

In [ ]:
# keep a copy of the result outside the session
import shutil
src = f'{WORK}/results/day1/cloud_{PLATFORM}.json'
dst = '/content/drive/MyDrive/mtp/' if PLATFORM == 'colab' else '/kaggle/working/'
shutil.copy(src, dst); print(open(src).read())